In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import roc_auc_score

# Reload the final 30-feature state
X_train_final = np.load('../data/X_train_final_30.npy')
X_test_final = np.load('../data/X_test_final_30.npy')
final_feature_names = np.load('../data/final_feature_names_30.npy', allow_pickle=True)

# Reload the dataframes (needed for patient_nbr and target)
df_train = pd.read_csv('../data/df_train_v2.csv')
df_test = pd.read_csv('../data/df_test_v2.csv')
y_train = df_train['target']
y_test = df_test['target']

# Verify alignment
assert len(df_train) == len(X_train_final), f"Mismatch: df_train {len(df_train)} vs X_train_final {len(X_train_final)}"

print(f"X_train_final: {X_train_final.shape}")
print(f"X_test_final:  {X_test_final.shape}")
print(f"df_train rows: {len(df_train):,}")
print(f"Positive rate: {y_train.mean()*100:.2f}%")

X_train_final: (78283, 30)
X_test_final:  (19539, 30)
df_train rows: 78,283
Positive rate: 11.33%


In [2]:
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

# We need patient_nbr aligned with X_train_final
# It's still in df_train; align indices
patient_groups = df_train['patient_nbr'].values

# Verify alignment
assert len(patient_groups) == len(X_train_final), "Patient groups must align with X_train"
print(f"Number of training patients (unique): {len(np.unique(patient_groups)):,}")
print(f"Number of training encounters: {len(patient_groups):,}")

# Set up 5-fold CV respecting patient groups
group_kfold = GroupKFold(n_splits=5)

# The model we're CV'ing — same as our final LR baseline
cv_model = LogisticRegression(
    penalty='l2',
    solver='lbfgs',
    C=1.0,
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

# Run cross-validation
cv_scores = cross_val_score(
    cv_model,
    X_train_final,
    y_train,
    groups=patient_groups,
    cv=group_kfold,
    scoring='roc_auc',
    n_jobs=-1  # use all CPU cores
)

print(f"\nCV AUC scores across 5 folds: {cv_scores}")
print(f"\nMean AUC:  {cv_scores.mean():.4f}")
print(f"Std Dev:   {cv_scores.std():.4f}")
print(f"95% CI:    [{cv_scores.mean() - 2*cv_scores.std():.4f}, {cv_scores.mean() + 2*cv_scores.std():.4f}]")
print(f"\nRange:     [{cv_scores.min():.4f}, {cv_scores.max():.4f}]")

Number of training patients (unique): 55,108
Number of training encounters: 78,283


/home/sharon/anaconda3/envs/datacareer/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/sharon/anaconda3/envs/datacareer/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/sharon/anaconda3/envs/datacareer/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in versi


CV AUC scores across 5 folds: [0.65718413 0.66443097 0.65867903 0.66196111 0.6610251 ]

Mean AUC:  0.6607
Std Dev:   0.0025
95% CI:    [0.6556, 0.6657]

Range:     [0.6572, 0.6644]
